In [1]:
from pathlib import Path
import pandas as pd
import torch
import re
import numpy as np

# Paths
obj_dir = Path('/home/jakaria/ADNI/ADNI_1/adni_processed/left_hippocampus_correspondence/minimal_scaled_obj_files')
metadata_csv = Path('/home/jakaria/ADNI/ADNI_1/adni_processed/adni_metadata_filtered.csv')
volume_csv = Path('/home/jakaria/ADNI/ADNI_1/adni_processed/adni_xml_metadata.csv')
output_path = obj_dir / 'labels.pt'

print(f"OBJ directory: {obj_dir}")
print(f"Metadata CSV: {metadata_csv}")
print(f"Volume CSV: {volume_csv}")
print(f"Output path: {output_path}")

OBJ directory: /home/jakaria/ADNI/ADNI_1/adni_processed/left_hippocampus_correspondence/minimal_scaled_obj_files
Metadata CSV: /home/jakaria/ADNI/ADNI_1/adni_processed/adni_metadata_filtered.csv
Volume CSV: /home/jakaria/ADNI/ADNI_1/adni_processed/adni_xml_metadata.csv
Output path: /home/jakaria/ADNI/ADNI_1/adni_processed/left_hippocampus_correspondence/minimal_scaled_obj_files/labels.pt


In [2]:
# Load metadata files
metadata_df = pd.read_csv(metadata_csv)
volume_df = pd.read_csv(volume_csv)

print(f"Metadata shape: {metadata_df.shape}")
print(f"Volume data shape: {volume_df.shape}")
print("\nMetadata columns:", metadata_df.columns.tolist())
print("\nVolume data columns:", volume_df.columns.tolist())
print("\nMetadata sample:")
print(metadata_df.head())
print("\nVolume data sample:")
print(volume_df.head())

Metadata shape: (1632, 6)
Volume data shape: (1632, 9)

Metadata columns: ['subject_id', 'diagnosis', 'gender', 'age', 'visit', 'image_data_id']

Volume data columns: ['filename', 'subject_id', 'research_group', 'subject_age', 'subject_sex', 'image_uid', 'left_hippocampus_volume', 'right_hippocampus_volume', 'total_hippocampus_volume']

Metadata sample:
   subject_id diagnosis gender  age visit image_data_id
0  002_S_0954       MCI      F   69    sc        I85433
1  005_S_0572       MCI      M   79    bl        I85434
2  005_S_0324       MCI      F   75    bl        I91387
3  005_S_0324       MCI      F   76   m06        I91388
4  005_S_0324       MCI      F   76   m12        I91389

Volume data sample:
                                          filename  subject_id research_group  \
0  ADNI_036_S_0869_Hippocampal_Mask_S31884_I100424  036_S_0869            MCI   
1  ADNI_130_S_0886_Hippocampal_Mask_S41101_I100559  130_S_0886             CN   
2  ADNI_128_S_0863_Hippocampal_Mask_S29526_I

In [3]:
# Get all left hippocampus OBJ files
obj_files = sorted([f.name for f in obj_dir.glob('*.obj') if 'left' in f.name.lower()])
print(f"Found {len(obj_files)} left hippocampus OBJ files")
print("\nSample filenames:")
for i in range(min(5, len(obj_files))):
    print(obj_files[i])

Found 1632 left hippocampus OBJ files

Sample filenames:
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111448800_S13408_I93328_left.obj
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111508581_S21856_I93329_left.obj
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111524597_S32678_I93331_left.obj
ADNI_002_S_0413_MR_Hippocampal_Mask_Hi_20090630133420824_S13893_I146878_left.obj
ADNI_002_S_0413_MR_Hippocampal_Mask_Hi_20090630133440731_S22557_I146879_left.obj


In [4]:
# Extract subject_id and image_id from filename
# Example: ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111448800_S13408_I93328_left.obj
# Pattern: ADNI_{subject_id}_MR_..._{series_id}_{image_id}_left.obj

def parse_filename(filename):
    """
    Extract subject_id and image_id from ADNI filename.
    Returns (subject_id, image_id) or (None, None) if parsing fails.
    """
    # Match pattern like: ADNI_002_S_0295_MR_..._I93328_left.obj
    pattern = r'ADNI_(\d+_S_\d+)_.*_I(\d+)_left\.obj'
    match = re.search(pattern, filename)
    
    if match:
        subject_id = match.group(1)
        image_id = match.group(2)
        return subject_id, image_id
    return None, None

# Test parsing
print("Testing filename parsing:")
for i in range(min(5, len(obj_files))):
    fname = obj_files[i]
    subj, img = parse_filename(fname)
    print(f"{fname} -> subject: {subj}, image: {img}")

Testing filename parsing:
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111448800_S13408_I93328_left.obj -> subject: 002_S_0295, image: 93328
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111508581_S21856_I93329_left.obj -> subject: 002_S_0295, image: 93329
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111524597_S32678_I93331_left.obj -> subject: 002_S_0295, image: 93331
ADNI_002_S_0413_MR_Hippocampal_Mask_Hi_20090630133420824_S13893_I146878_left.obj -> subject: 002_S_0413, image: 146878
ADNI_002_S_0413_MR_Hippocampal_Mask_Hi_20090630133440731_S22557_I146879_left.obj -> subject: 002_S_0413, image: 146879


In [5]:
# Create mapping from image_id to metadata
# Assuming metadata has 'image_id' and volume data has 'image_id' columns

# Check what columns exist for matching
print("Checking for matching columns...")
print(f"Metadata columns with 'image' or 'id': {[c for c in metadata_df.columns if 'image' in c.lower() or 'id' in c.lower()]}")
print(f"Volume columns with 'image' or 'id': {[c for c in volume_df.columns if 'image' in c.lower() or 'id' in c.lower()]}")
print(f"Volume columns with 'hippo' or 'volume': {[c for c in volume_df.columns if 'hippo' in c.lower() or 'volume' in c.lower()]}")

Checking for matching columns...
Metadata columns with 'image' or 'id': ['subject_id', 'image_data_id']
Volume columns with 'image' or 'id': ['subject_id', 'image_uid']
Volume columns with 'hippo' or 'volume': ['left_hippocampus_volume', 'right_hippocampus_volume', 'total_hippocampus_volume']


In [6]:
# Create labels dictionary
labels = {}
missing_metadata = []
missing_volume = []
successfully_processed = []

# Map diagnosis to binary: CN=0, AD=1
diagnosis_map = {'CN': 0, 'AD': 1}

# Map sex to numeric: M=0, F=1
sex_map = {'M': 0, 'F': 1}

for fname in obj_files:
    base_name = fname.replace('.obj', '')
    subject_id, image_id = parse_filename(fname)
    
    if subject_id is None or image_id is None:
        print(f"Warning: Could not parse filename {fname}")
        continue
    
    # Match with metadata (use subject_id and potentially image_id)
    # First, filter metadata by subject_id
    subject_metadata = metadata_df[metadata_df['subject_id'] == subject_id]
    
    if len(subject_metadata) == 0:
        missing_metadata.append(fname)
        continue
    
    # If there are multiple rows for a subject, try to match by image_id
    # Check if image_id column exists
    if 'image_id' in subject_metadata.columns:
        image_metadata = subject_metadata[subject_metadata['image_id'].astype(str) == image_id]
        if len(image_metadata) > 0:
            subject_metadata = image_metadata
    
    # Take the first matching row
    metadata_row = subject_metadata.iloc[0]
    
    # Extract diagnosis, age, sex
    diagnosis_str = metadata_row.get('diagnosis', 'MCI')  # Default to MCI if missing
    diagnosis = diagnosis_map.get(diagnosis_str, -1)  # -1 for MCI or unknown
    
    age = metadata_row.get('age', np.nan)
    sex_str = metadata_row.get('sex', 'U')  # U for unknown
    sex = sex_map.get(sex_str, -1)
    
    # Match with volume data
    # Filter by subject_id and image_id
    volume_match = volume_df[(volume_df['subject_id'] == subject_id)]
    
    if 'image_id' in volume_df.columns:
        volume_match = volume_match[volume_match['image_id'].astype(str) == image_id]
    
    if len(volume_match) == 0:
        missing_volume.append(fname)
        volume = np.nan
    else:
        volume_row = volume_match.iloc[0]
        # Look for left hippocampus volume column
        volume_col = [c for c in volume_df.columns if 'left' in c.lower() and 'hippo' in c.lower()]
        if len(volume_col) > 0:
            volume = volume_row.get(volume_col[0], np.nan)
        else:
            volume = np.nan
    
    # Create label tensor: [diagnosis, age, sex, volume]
    label = torch.tensor([diagnosis, age, sex, volume], dtype=torch.float32)
    labels[base_name] = label
    successfully_processed.append(fname)

print(f"\nProcessing summary:")
print(f"Total OBJ files: {len(obj_files)}")
print(f"Successfully processed: {len(successfully_processed)}")
print(f"Missing metadata: {len(missing_metadata)}")
print(f"Missing volume data: {len(missing_volume)}")


Processing summary:
Total OBJ files: 1632
Successfully processed: 1632
Missing metadata: 0
Missing volume data: 0


In [7]:
# Display sample labels
print("\nSample labels:")
for i, (key, value) in enumerate(labels.items()):
    if i < 10:
        print(f"{key}: {value.tolist()} (diagnosis={value[0]}, age={value[1]}, sex={value[2]}, volume={value[3]})")
    else:
        break


Sample labels:
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111448800_S13408_I93328_left: [0.0, 85.0, -1.0, 2144.155517578125] (diagnosis=0.0, age=85.0, sex=-1.0, volume=2144.155517578125)
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111508581_S21856_I93329_left: [0.0, 85.0, -1.0, 2144.155517578125] (diagnosis=0.0, age=85.0, sex=-1.0, volume=2144.155517578125)
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111524597_S32678_I93331_left: [0.0, 85.0, -1.0, 2144.155517578125] (diagnosis=0.0, age=85.0, sex=-1.0, volume=2144.155517578125)
ADNI_002_S_0413_MR_Hippocampal_Mask_Hi_20090630133420824_S13893_I146878_left: [0.0, 76.0, -1.0, 2095.460205078125] (diagnosis=0.0, age=76.0, sex=-1.0, volume=2095.460205078125)
ADNI_002_S_0413_MR_Hippocampal_Mask_Hi_20090630133440731_S22557_I146879_left: [0.0, 76.0, -1.0, 2095.460205078125] (diagnosis=0.0, age=76.0, sex=-1.0, volume=2095.460205078125)
ADNI_002_S_0413_MR_Hippocampal_Mask_Hi_20090630133455856_S32938_I146880_left: [0.0, 76.0, -1.0, 2095.4

In [8]:
# Save labels
torch.save(labels, output_path)
print(f"\nSaved {len(labels)} labels to {output_path}")


Saved 1632 labels to /home/jakaria/ADNI/ADNI_1/adni_processed/left_hippocampus_correspondence/minimal_scaled_obj_files/labels.pt


In [9]:
# Verification: Load and check saved labels
loaded_labels = torch.load(output_path)
print(f"\nVerification: Loaded {len(loaded_labels)} labels from {output_path}")

# Check a few random samples
sample_keys = list(loaded_labels.keys())[:5]
print("\nVerifying sample labels:")
for key in sample_keys:
    label = loaded_labels[key]
    print(f"{key}: diagnosis={label[0]}, age={label[1]}, sex={label[2]}, volume={label[3]}")


Verification: Loaded 1632 labels from /home/jakaria/ADNI/ADNI_1/adni_processed/left_hippocampus_correspondence/minimal_scaled_obj_files/labels.pt

Verifying sample labels:
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111448800_S13408_I93328_left: diagnosis=0.0, age=85.0, sex=-1.0, volume=2144.155517578125
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111508581_S21856_I93329_left: diagnosis=0.0, age=85.0, sex=-1.0, volume=2144.155517578125
ADNI_002_S_0295_MR_Hippocampal_Mask_Hi_20080228111524597_S32678_I93331_left: diagnosis=0.0, age=85.0, sex=-1.0, volume=2144.155517578125
ADNI_002_S_0413_MR_Hippocampal_Mask_Hi_20090630133420824_S13893_I146878_left: diagnosis=0.0, age=76.0, sex=-1.0, volume=2095.460205078125
ADNI_002_S_0413_MR_Hippocampal_Mask_Hi_20090630133440731_S22557_I146879_left: diagnosis=0.0, age=76.0, sex=-1.0, volume=2095.460205078125


In [10]:
# Statistics
all_labels_array = torch.stack(list(loaded_labels.values()))

print("\nLabel statistics:")
print(f"Diagnosis distribution: CN={torch.sum(all_labels_array[:, 0] == 0).item()}, AD={torch.sum(all_labels_array[:, 0] == 1).item()}")
print(f"Age range: {torch.min(all_labels_array[:, 1]):.1f} - {torch.max(all_labels_array[:, 1]):.1f}")
print(f"Age mean: {torch.mean(all_labels_array[:, 1]):.1f}")
print(f"Sex distribution: M={torch.sum(all_labels_array[:, 2] == 0).item()}, F={torch.sum(all_labels_array[:, 2] == 1).item()}")
print(f"Volume range: {torch.min(all_labels_array[:, 3]):.1f} - {torch.max(all_labels_array[:, 3]):.1f}")
print(f"Volume mean: {torch.mean(all_labels_array[:, 3]):.1f}")


Label statistics:
Diagnosis distribution: CN=477, AD=342
Age range: 55.0 - 90.0
Age mean: 75.6
Sex distribution: M=0, F=0
Volume range: 816.8 - 2807.9
Volume mean: 1829.2


In [11]:
# Cross-reference with split files to ensure all training/test files have labels
import json

split_dir = Path('../examples/splits/splits_left_hippocampus_ADNI_No_MCI')
train_split_file = split_dir / 'train_split_left_hippocampus_adni_no_mci.json'
test_split_file = split_dir / 'test_split_left_hippocampus_adni_no_mci.json'

if train_split_file.exists() and test_split_file.exists():
    with open(train_split_file, 'r') as f:
        train_files = json.load(f)
    with open(test_split_file, 'r') as f:
        test_files = json.load(f)
    
    all_split_files = train_files + test_files
    missing_in_labels = []
    
    for fname in all_split_files:
        base_name = fname.replace('.obj', '')
        if base_name not in loaded_labels:
            missing_in_labels.append(fname)
    
    print(f"\nSplit file verification:")
    print(f"Total files in splits: {len(all_split_files)}")
    print(f"Files in splits missing labels: {len(missing_in_labels)}")
    
    if missing_in_labels:
        print("\nMissing labels for:")
        for fname in missing_in_labels[:10]:
            print(f"  {fname}")
else:
    print(f"\nSplit files not found at {split_dir}")


Split file verification:
Total files in splits: 773
Files in splits missing labels: 0
